In [1]:
import pandas as pd
import numpy as np

# E-Commerce Dataset cleaning 

The primary purpose of this section of the project is to clean the E-commerce dataset,
in preparation for Recency-Frequency-Monetary(RFM) Analysis in the next section of the project. 
The data cleaning process focuses on investigating the data quality problems,
which includes: 

- Checking for and replacing missing values where neccessary,
- Identification and removal of duplicated rows of records in the dataset,
- Rectifying incorrect datatypes such as dates,strings, float and integers,
- Save the final cleaned dataset as csv file.  

# Importing and checking the dataset structure.

In [2]:
data = pd.read_csv('ecommerce_sales_customer_analytics_150k.csv')

In [3]:
data.shape

(138116, 46)

In [4]:
data.head()

,order_id,order_date,order_time,order_status,sales_channel,customer_id,customer_name,customer_age,gender,customer_segment,...,discount_amount,tax_amount,shipping_cost,net_sales,product_cost,profit,profit_margin_percentage,customer_lifetime_value,is_repeat_customer,customer_order_count
0,ORD-301242,2023-11-06,16:37:47,Completed,Mobile App,CUST-003102,Jasmine Ryan,55,Male,Consumer,...,477.89,61.07,12.03,945.40,588.33,345.04,36.50,12459.68,True,11
1,ORD-773460,2025-12-24,01:22:36,Completed,Website,CUST-003124,Scott Chase,26,Male,Premium,...,1456.06,320.83,9.00,2018.41,2031.88,-22.47,-1.11,13032.48,True,11
2,ORD-449374,2021-07-05,14:24:21,Completed,Mobile App,CUST-012496,Marc Wheeler,69,Female,Premium,...,97.30,29.79,13.05,468.35,257.73,197.57,42.18,6159.44,True,6
3,ORD-567636,2023-01-21,07:20:26,Completed,Social Media,CUST-023928,Jennifer Smith,65,Male,Consumer,...,53.34,34.39,20.85,546.52,277.84,247.83,45.35,5638.30,True,7
4,ORD-820028,2022-05-13,09:46:21,Completed,Social Media,CUST-012730,Jessica Wang,34,Female,Consumer,...,133.63,421.35,23.03,2785.27,1231.88,1530.36,54.94,13240.10,True,8


In [5]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 138116 entries, 0 to 138115
Data columns (total 46 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   order_id                  138116 non-null  str    
 1   order_date                138116 non-null  str    
 2   order_time                138116 non-null  str    
 3   order_status              138116 non-null  str    
 4   sales_channel             138116 non-null  str    
 5   customer_id               138116 non-null  str    
 6   customer_name             138116 non-null  str    
 7   customer_age              138116 non-null  int64  
 8   gender                    138116 non-null  str    
 9   customer_segment          138116 non-null  str    
 10  customer_type             138116 non-null  str    
 11  customer_city             138116 non-null  str    
 12  customer_state            138116 non-null  str    
 13  customer_country          138116 non-null  str    
 14 

# 1. Handling missing values

In [6]:
data.isnull().sum()

order_id                         0
order_date                       0
order_time                       0
order_status                     0
sales_channel                    0
customer_id                      0
customer_name                    0
customer_age                     0
gender                           0
customer_segment                 0
customer_type                    0
customer_city                    0
customer_state                   0
customer_country                 0
region                           0
customer_postal_code             0
payment_method                   0
payment_status                   0
currency                         0
shipping_method                  0
warehouse                        0
delivery_days                24557
estimated_delivery_days      24557
delivery_status                  0
return_status               128654
return_reason               128654
customer_rating              24557
review_sentiment             24557
customer_review     

### Missing Values Dataframe

In [7]:
nulls_df = pd.DataFrame({'Nulls':data.isnull().sum(), 'Nulls_%':round(100*(data.isnull().sum()/len(data['shipping_cost'])),2)})

In [8]:
nulls_df = nulls_df[nulls_df['Nulls']>0].sort_values(by='Nulls').reset_index().rename({'index':'Colomn_Name'},axis = 1)

In [9]:
nulls_df

,Colomn_Name,Nulls,Nulls_%
0,delivery_days,24557,17.78
1,estimated_delivery_days,24557,17.78
2,review_sentiment,24557,17.78
3,customer_rating,24557,17.78
4,customer_review,24557,17.78
5,campaign_name,83233,60.26
6,coupon_code,110502,80.01
7,return_reason,128654,93.15
8,return_status,128654,93.15


- The dataset appears to have equal number of missing values in the columns **return_reason** and **return_status**, about **93.15%**;
- Followed by **coupon_code** and **campaign_name** about **80.01%** and **60.26%** respectively.
- We also have equal number of missing values in the columns (**delivery_days**, **estimated_delivery_days**,  **review_setiment**, **customer_rating**, **customer_review**), approximatley **17.78%**

## 1.2 Handling missing values in the columns; **return_status** and **return_reason**.
- Since we have equal numnber of missing values in these two columns, we may assume that these missing values are related somehow.
- we are going to handle these missing values by studying their relationship with the **order_status** column.

In [10]:
print(f'return_status missing values: {data['return_status'].isnull().sum()}')

return_status missing values: 128654


In [11]:
print(f'return_reason missing values: {data['return_reason'].isnull().sum()}')

return_reason missing values: 128654


In [12]:
data['order_status'].value_counts(dropna=False)

order_status
Completed    113559
Returned       9462
Cancelled      8398
Pending        6697
Name: count, dtype: int64

**return_status and return_reason missing values counts corresponding to Completed order_status**

In [13]:
data[data['order_status'] =='Completed'][['order_status','return_status','return_reason']].value_counts(dropna = False)

order_status  return_status  return_reason
Completed     NaN            NaN              113559
Name: count, dtype: int64

**return_status and return_reason missing values counts corresponding to Cancelled order_status**

In [14]:
data[data['order_status'] =='Cancelled'][['order_status','return_status','return_reason']].value_counts(dropna = False)

order_status  return_status  return_reason
Cancelled     NaN            NaN              8398
Name: count, dtype: int64

**return_status and return_reason missing values counts corresponding to Pending order_status**

In [15]:
data[data['order_status'] =='Pending'][['order_status','return_status','return_reason']].value_counts(dropna = False)

order_status  return_status  return_reason
Pending       NaN            NaN              6697
Name: count, dtype: int64

In [16]:
6697+113559+8398

128654

- Its clear that all the 128654 NaN values in the **return_status** column correspond with the *"Completed"*, *"Pending"* and *"Cancelled"* **order_status**.
- These Nan values may be replaced with *"Not_Returned"*, since an order may either be *"returned"* or *"Not_returned"*.
- Similarly all the 128654 Nan Values in the **return_reason** column correspond with the *"Completed"*, *"Pending"* and *"Cancelled"* **order_status**.
- These NaN values can be replaced with not applicable as 'n\a', since no valid return_reason can be given for *"Completed"*, *"Pending"* or *"Cancelled"* orders

In [17]:
data = data.fillna({'return_status':'Not_cancelled','return_reason':'Not_Applicable'})

In [18]:
data['return_status'].value_counts(dropna =False)

return_status
Not_cancelled    128654
Returned           9462
Name: count, dtype: int64

In [19]:
data['return_reason'].value_counts(dropna =False)

return_reason
Not_Applicable             128654
Wrong Product                1237
Other                        1208
Changed Mind                 1204
Size Issue                   1182
Defective Product            1174
Late Delivery                1172
Product Not as Expected      1147
Damaged Product              1138
Name: count, dtype: int64

## 1.3 Handling missing values in the **coupon_code** column

In [20]:
data['coupon_code'].value_counts(dropna = False).head()

coupon_code
NaN       110502
SAVE39       731
SAVE49       727
SAVE40       719
SAVE12       716
Name: count, dtype: int64

- The missing values on the **coupon_code** column appear to be from customers who had no coupon to spend or chose not to spend thier coupon when placing orders.
- Since there is no way of knowing whether these customers had no coupons or chose not to spend thier coupons, it would be ideal to use *"No_Coupon"* to replace these missing or NaN values.

In [21]:
data = data.fillna({'coupon_code':'No_Coupon'})

In [25]:
data['coupon_code'].value_counts(dropna = False).head()

coupon_code
No_Coupon    110502
SAVE39          731
SAVE49          727
SAVE40          719
SAVE12          716
Name: count, dtype: int64

## 1.4 Handling missing values in the columns **review_setiment**, **customer_rating** and **customer_review**

- Since we have equal number of missing values in these three columns, we may assume that these missing values are somehow related.
- we are going to handle these missing values by studying their relationship with the **order_status** column.

In [23]:
nulls_df[2:5]

,Colomn_Name,Nulls,Nulls_%
2,review_sentiment,24557,17.78
3,customer_rating,24557,17.78
4,customer_review,24557,17.78


In [24]:
data['order_status'].value_counts(dropna=False)

order_status
Completed    113559
Returned       9462
Cancelled      8398
Pending        6697
Name: count, dtype: int64

**review_sentiment, customer_review , customer_rating missing value counts corresponding to Pending order_status**

In [25]:
data[data['order_status'] =='Pending'][['order_status','review_sentiment','customer_review','customer_rating']].value_counts(dropna = False)

order_status  review_sentiment  customer_review  customer_rating
Pending       NaN               NaN              NaN                6697
Name: count, dtype: int64

- This implies that 6697 NaN values in **review_sentiment**, **customer_review** and **customer_rating** come from all the 6697 *"Pending"* orders
- That is: All the customers with pending orders did not write any review and therefore no customer rating or review sentiment would be applicable.

**review_sentiment, customer_review , customer_rating missing value counts corresponding to Cancelled order_status**

In [26]:
data[data['order_status'] =='Cancelled'][['order_status','review_sentiment','customer_review','customer_rating']].value_counts(dropna = False)

order_status  review_sentiment  customer_review  customer_rating
Cancelled     NaN               NaN              NaN                8398
Name: count, dtype: int64

- This implies that 8398 NaN values in **review_sentiment**,  **customer_review** and **customer_rating** come from all the 8398 *"Cancelled"* orders.
- That is: All the customers with cancelled orders did not write any review and therefore no customer rating or review sentiment would be applicable.

**review_sentiment, customer_review , customer_rating missing value counts corresponding to Returned order_status**

In [27]:
data[data['order_status'] =='Returned'][['order_status','review_sentiment','customer_review','customer_rating']].value_counts(dropna = False)

order_status  review_sentiment  customer_review  customer_rating
Returned      NaN               NaN              NaN                9462
Name: count, dtype: int64

- This implies that 9462 NaN values in **review_sentiment**,  **customer_review** and **customer_rating** come from all the 9462 *"Returned"* orders
- That is: All the customers who returned their orders did not write any review and therefore no customer rating or review sentiment would be applicable.

In [26]:
6697+8398+9462

24557

- It is evident that all the 24557 missing values in each of the columns (**customer_review**,**customer_rating**,**review_sentiment**) are equivalent to the sum of **order_status** corresponding to *"Pending"*, *"Cancelled"* and *"Returned"*.
- Since the customer rating and review sentiment follows directly from the customer review, it makes sense to assume that:
- The missing values on the **customer review** be replaced by *"No_Review"*, and
- The corresponding missing values on **customer_rating** and **review_sentiment be replaced** by *"Not_Applicable"*.

In [27]:
data = data.fillna({'customer_review':'No_Review','customer_rating':'Not_Applicable','review_sentiment':'Not_Applicable'})

In [28]:
data[['review_sentiment','customer_review','customer_rating']].isnull().sum()

review_sentiment    0
customer_review     0
customer_rating     0
dtype: int64

## 1.5 Handling the missing values in the columns **delivery_days** and **estimated_delivery_days**

In [29]:
nulls_df[0:2]

,Colomn_Name,Nulls,Nulls_%
0,delivery_days,24557,17.78
1,estimated_delivery_days,24557,17.78


In [30]:
data[data['order_status'] !='Completed'][['order_status','delivery_days','estimated_delivery_days']].value_counts(dropna = False)

order_status  delivery_days  estimated_delivery_days
Returned      NaN            NaN                        9462
Cancelled     NaN            NaN                        8398
Pending       NaN            NaN                        6697
Name: count, dtype: int64

In [31]:
9462+8398+6697

24557

- It is evident that the same technique used to handle the customer reviews can be used to handle the missing values for  **delivery_days** and **estimated_delivery_days** columns.
- That is:
- All *"Returned"*, *"Cancelled"* and *"Pending"* orders have no corresponding values in the  **delivery_days** and **estimated_delivery_days** columns.
- since the delivery will never happen for a cancelled order, it makes sense to use *"Not_Applicable"* to replace missing values for both **delivery_days** and **estimated_delivery_days**,
- Whereas for *"Pending"* orders, *"Pending"* may be used since the delivery will be scheduled in the near future.
- Since the *Returned* orders may either be refunded or rescheduled for new delivery, *"Not_Applicable"* or *"Pending"* may be used respectively for the missing values in both **delivery_days** and **estimated_delivery_days**.
- For simplicity I would assume all returned orders are refunded, and hence use *"Not_Applicable"* to replace the missing values in both  **delivery_days** and **estimated_delivery_days**.

**delivery and estimated delivery days for returned or cancelled orders are replced by Not_Applicable**

In [32]:
condition = (data['order_status']=='Returned') | (data['order_status']=='Cancelled')

In [33]:
data['delivery_days'] = np.where(condition,'Not_Applicable',data['delivery_days'])

In [34]:
data['estimated_delivery_days'] = np.where(condition,'Not_Applicable',data['estimated_delivery_days'])

**delivery and estimated delivery days for Pending orders are replced by Pending**

In [35]:
data['delivery_days'] = np.where(data['order_status']=='Pending','Pending',data['delivery_days'])

In [36]:
data['estimated_delivery_days'] = np.where(data['order_status']=='Pending','Pending',data['estimated_delivery_days'])

In [37]:
data[['delivery_days','estimated_delivery_days']].isnull().sum()

delivery_days              0
estimated_delivery_days    0
dtype: int64

## 1.6 Handling missing Values in the **Campaign_Name** column

- The **campaign_name** missing values can be handled by exploiting the relationship with the **marketing_channel** column. 

In [40]:
print(data['campaign_name'].isnull().sum())

83233


In [41]:
data.groupby('marketing_channel')['campaign_name'].value_counts(dropna=False)

marketing_channel  campaign_name     
Affiliate          NaN                    4105
                   Influencer_Program     1404
                   Affiliate_Partner      1339
Direct             NaN                   12333
                   Default_Campaign       8182
Email Marketing    NaN                    6780
                   Abandoned_Cart         1520
                   Weekly_Newsletter      1459
                   Promo_Email            1459
Facebook Ads       NaN                    9987
                   FB_Retargeting         2270
                   FB_Dynamic             2233
                   FB_Prospecting         2161
Google Ads         NaN                   12408
                   Google_Search_Q1       2748
                   Google_Shopping        2746
                   Google_Display         2731
Instagram          NaN                    8407
                   Insta_Influencer       1822
                   Insta_Reels            1799
                   Ins

- The **marketing_channels** *"YouTube"*, *"TikTok"*, *"Organic_Search"* and *Direct* have only (*"Default_Campaign"* and *"NaN"*) as **campaign_name**.
- It is reasonable to assume that there was a data quality issue, and these marketing channels were supposed to be classified as defualt camapaign.
- Therefore all missing values in the **campaign_name** corresponding to **marketing_channels**: *"YouTube"*, *"TikTok"*, *"Organic_Search"* and *Direct*, can be replaced by *"Default_Campaign"*

- All the remaining seven **marketing_channels** have three or more **campaign_name** including the NaN values.
- And therefore, it is not possible to extrapolate the missing **campaign_names** corresponding these **marketing_channels**.
- For simplicity the missing values in the **campaign_name** corresponding these **marketing_channels** can be replaced by *"Uknown"*

In [38]:
condition = (data['marketing_channel'] == 'YouTube') | (data['marketing_channel'] == 'TikTok') |(data['marketing_channel'] == 'Organic_Search') | (data['marketing_channel'] == 'Direct')

In [39]:
data['campaign_name'] = np.where(condition,'Default_Campaign',data['campaign_name'])

In [40]:
data['campaign_name'] = data['campaign_name'].fillna('Uknown')

In [41]:
print(data['campaign_name'].isnull().sum())

0


## 1.6 Checking if all missing/NaN values had been handled

In [42]:
data.isnull().sum()

order_id                    0
order_date                  0
order_time                  0
order_status                0
sales_channel               0
customer_id                 0
customer_name               0
customer_age                0
gender                      0
customer_segment            0
customer_type               0
customer_city               0
customer_state              0
customer_country            0
region                      0
customer_postal_code        0
payment_method              0
payment_status              0
currency                    0
shipping_method             0
warehouse                   0
delivery_days               0
estimated_delivery_days     0
delivery_status             0
return_status               0
return_reason               0
customer_rating             0
review_sentiment            0
customer_review             0
marketing_channel           0
campaign_name               0
coupon_code                 0
loyalty_points_earned       0
loyalty_po

- **All missing values had been handled and replaced with appropriate values.**

# 2. Data types handlings 

In [43]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 138116 entries, 0 to 138115
Data columns (total 46 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   order_id                  138116 non-null  str    
 1   order_date                138116 non-null  str    
 2   order_time                138116 non-null  str    
 3   order_status              138116 non-null  str    
 4   sales_channel             138116 non-null  str    
 5   customer_id               138116 non-null  str    
 6   customer_name             138116 non-null  str    
 7   customer_age              138116 non-null  int64  
 8   gender                    138116 non-null  str    
 9   customer_segment          138116 non-null  str    
 10  customer_type             138116 non-null  str    
 11  customer_city             138116 non-null  str    
 12  customer_state            138116 non-null  str    
 13  customer_country          138116 non-null  str    
 14 

- The columns **order_date** and **order_time** were stored as string data type.
- These columns must be transformed to datetime data type.  

In [44]:
import datetime

In [45]:
data[['order_date','order_time']].head(2)

,order_date,order_time
0,2023-11-06,16:37:47
1,2025-12-24,01:22:36


In [50]:
data['order_date'] = pd.to_datetime(data['order_date'], format='%Y-%m-%d')
data['order_time'] = pd.to_datetime(data['order_time'], format='%H:%M:%S').dt.time

In [51]:
data['order_time'][1]

datetime.time(1, 22, 36)

In [52]:
data['order_date'][1]

Timestamp('2025-12-24 00:00:00')

## 3. Check and remove any duplicated rows 

In [46]:
data[data.duplicated()]

,order_id,order_date,order_time,order_status,sales_channel,customer_id,customer_name,customer_age,gender,customer_segment,...,discount_amount,tax_amount,shipping_cost,net_sales,product_cost,profit,profit_margin_percentage,customer_lifetime_value,is_repeat_customer,customer_order_count


- **it appears that there are no duplicated rows in the dataset.**

# 4. Saving the final cleaned dataset as **Cleaned_E_Commerce.csv**

In [47]:
data.to_csv('Cleaned_E_Commerce.csv',index=False)

In [48]:
df = pd.read_csv('Cleaned_E_Commerce.csv').head()

In [49]:
df.head()

,order_id,order_date,order_time,order_status,sales_channel,customer_id,customer_name,customer_age,gender,customer_segment,...,discount_amount,tax_amount,shipping_cost,net_sales,product_cost,profit,profit_margin_percentage,customer_lifetime_value,is_repeat_customer,customer_order_count
0,ORD-301242,2023-11-06,16:37:47,Completed,Mobile App,CUST-003102,Jasmine Ryan,55,Male,Consumer,...,477.89,61.07,12.03,945.40,588.33,345.04,36.50,12459.68,True,11
1,ORD-773460,2025-12-24,01:22:36,Completed,Website,CUST-003124,Scott Chase,26,Male,Premium,...,1456.06,320.83,9.00,2018.41,2031.88,-22.47,-1.11,13032.48,True,11
2,ORD-449374,2021-07-05,14:24:21,Completed,Mobile App,CUST-012496,Marc Wheeler,69,Female,Premium,...,97.30,29.79,13.05,468.35,257.73,197.57,42.18,6159.44,True,6
3,ORD-567636,2023-01-21,07:20:26,Completed,Social Media,CUST-023928,Jennifer Smith,65,Male,Consumer,...,53.34,34.39,20.85,546.52,277.84,247.83,45.35,5638.30,True,7
4,ORD-820028,2022-05-13,09:46:21,Completed,Social Media,CUST-012730,Jessica Wang,34,Female,Consumer,...,133.63,421.35,23.03,2785.27,1231.88,1530.36,54.94,13240.10,True,8


**The Cleaned_E_Commerce.csv file has been saved successfully and ready for the section of the project.**